# Environment Setup

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "orb-models==0.5.5", "ase>=3.24", "numpy>=1.26", "scipy>=1.15", "tqdm>=4.66", "cached_path>=1.6.7", "dm-tree==0.1.8", "pandas>=2.2", "h5py>=3.11", "matplotlib>=3.9"])


In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "cudf", "cuml", "libcugraph-cu12", "pylibcugraph-cu12", "pylibraft-cu12", "libraft-cu12", "rmm-cu12"])


In [ ]:
# Data is expected under ../data/raw (see data/README.md)


# Params

In [ ]:
# Model size choices: tiny -> orb-d3-xs-v2, small -> orb-d3-sm-v2, full -> orb-v2
from pathlib import Path

MODEL_SIZE = "tiny"

# Sampling / data (set your dataset paths here)
RAW_DEEPMD_DIRS = [
    "../data/raw/beta-Li3PS4",
    "../data/raw/gamma-Li3PS4",
]
# If you already have XYZ files, set RAW_DEEPMD_DIRS = [] and fill RAW_DATASETS.
RAW_DATASETS = [f"../data/raw/{Path(path).name}.xyz" for path in RAW_DEEPMD_DIRS]

SAMPLE_SIZE = 1250  # frames per dataset for prepare_datasets

# Names inferred from file stems (override DATASET_NAMES if desired)
DATASET_NAMES = [Path(path).stem for path in RAW_DATASETS]
COMBINED_NAME = "_".join(DATASET_NAMES)

# Training hyperparams (defaults)
DEFAULT_EPOCHS = 100
DEFAULT_LR = 5e-5
DEFAULT_WEIGHT_DECAY = 3e-4
BATCH_SIZE = 16
VAL_FRAC = 0.1
TEST_FRAC = 0.1
SPLIT_SEED = 42
TRAIN_SEED = 42
CSV_SUFFIX = f"_seed{TRAIN_SEED}"
DATA_UNITS = "eV"  # dataset units; converted to eV for training/eval

# Per-model base training overrides (set only what differs)
BASE_TRAIN_OVERRIDES = {
    # Example:
    # "Li3P": {"epochs": 120, "lr": 1e-4},
    # "Li4P2S6": {"epochs": 80, "lr": 5e-5},
    # "combined": {"epochs": 150, "lr": 8e-5},
}

# Force-head fine-tune (swap embeddings)
FT_SAMPLES = 1024
FT_EPOCHS = 100
FT_LR = 5e-3
FT_SCHED = "none"
FT_SCHED_GAMMA = 0.5
FT_SCHED_PATIENCE = 4

# Fine-tune validation controls
FT_VAL_EVERY = 1   # 0 disables per-epoch val checks
FT_VAL_LIMIT = 0   # 0 uses full val split
FT_NO_VAL = False  # True disables val loss completely

# Stacks+head fine-tune
STACKS = 1          # 1/2/3/4
STACKS_LR = 5e-4    # backbone LR
STACKS_EPOCHS = 100
STACKS_SAMPLES = 512
STACKS_SCHED = "none"
STACKS_VAL_FRAC = None  # override val fraction for stacks fine-tune; set 0 to disable

# Toggle AMP for speed on GPU (doesn't work correctly for now)
USE_AMP = False 



In [ ]:
import os, subprocess, shlex, sys, time, re

os.environ.setdefault("PYTHONUNBUFFERED", "1")

def run(cmd):
    cmd_list = shlex.split(cmd) if isinstance(cmd, str) else list(cmd)
    print("\n>>>", " ".join(cmd_list))
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        sys.stdout.write(line)
    ret = proc.wait()
    if ret:
        raise SystemExit(ret)

def run_capture(cmd):
    cmd_list = shlex.split(cmd) if isinstance(cmd, str) else list(cmd)
    print("\n>>>", " ".join(cmd_list))
    proc = subprocess.Popen(cmd_list, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    output = []
    for line in proc.stdout:
        sys.stdout.write(line)
        output.append(line)
    ret = proc.wait()
    if ret:
        raise SystemExit(ret)
    return "".join(output)

def parse_logged_time(text, pattern):
    match = re.search(pattern, text, re.M)
    if match:
        return float(match.group(1))
    return None



# Prepare Datasets

In [ ]:
import time, json
from pathlib import Path

if RAW_DEEPMD_DIRS:
    cmd = [
        "python", "-u", "../scripts/convert_deepmd_raw_to_xyz.py",
        "--output-dir", "../data/raw",
    ]
    for raw_dir in RAW_DEEPMD_DIRS:
        name = Path(raw_dir).name
        cmd += ["--dataset", f"{name}={raw_dir}"]
    run(cmd)

cmd = [
    "python", "-u", "../scripts/prepare_datasets.py",
    "--combined-name", COMBINED_NAME,
    "--sample-size", str(SAMPLE_SIZE),
    "--output-dir", "../data/prepared",
]
for name, path in zip(DATASET_NAMES, RAW_DATASETS):
    cmd += ["--dataset", f"{name}={path}"]
start = time.time()
run(cmd)
print(f"prep_time_sec={int(time.time() - start)}")

summary_path = Path("../data/prepared/sampling_summary.json")
summary = json.loads(summary_path.read_text())

def dataset_entry(name: str):
    return next(entry for entry in summary["datasets"] if entry["name"] == name)

DATASET_DBS = {name: Path(dataset_entry(name)["db_path"]) for name in DATASET_NAMES}
COMBINED_DB = Path(dataset_entry(COMBINED_NAME)["db_path"])
for name in DATASET_NAMES:
    print(f"Dataset: {name} -> {DATASET_DBS[name]}")
print(f"Combined: {COMBINED_NAME} -> {COMBINED_DB}")



In [ ]:
import json
from pathlib import Path

# Ensure config JSONs exist for pretrained checkpoints (no training required)
MODEL_SIZE_TO_BASE = {
    "tiny": "orb-d3-xs-v2",
    "small": "orb-d3-sm-v2",
    "full": "orb-v2",
}
BASE_MODEL = MODEL_SIZE_TO_BASE[MODEL_SIZE]


def ensure_config(name: str, db_path: Path) -> None:
    out_dir = Path(f"../models/trained/{name}_{MODEL_SIZE}")
    out_dir.mkdir(parents=True, exist_ok=True)
    cfg_path = out_dir / f"{name}_config.json"
    if cfg_path.exists():
        print(f"Config exists: {cfg_path}")
        return
    cfg = {
        "dataset_name": name,
        "db_path": str(db_path),
        "dataset_path": str(db_path),
        "model_size": MODEL_SIZE,
        "base_model": BASE_MODEL,
        "precision": "float32-high",
        "data_units": DATA_UNITS,
        "val_fraction": VAL_FRAC,
        "test_fraction": TEST_FRAC,
        "seed": TRAIN_SEED,
        "split_seed": SPLIT_SEED,
        "batch_size": BATCH_SIZE,
        "force_only": True,
        "num_workers": 0,
    }
    cfg_path.write_text(json.dumps(cfg, indent=2))
    print(f"Wrote {cfg_path}")


for name in DATASET_NAMES:
    ensure_config(name, DATASET_DBS[name])
ensure_config(COMBINED_NAME, COMBINED_DB)



# Training (skipped: using pre-trained checkpoints)


In [ ]:
TRAIN_TIMES = {}
print("Skipping training: using pre-trained checkpoints in ../models/trained/")


# Evaluation of Trained Models

In [ ]:
def eval_model(name):
    cfg = f"../models/trained/{name}_{MODEL_SIZE}/{name}_config.json"
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    run([
        "python", "-u", "../scripts/evaluate_model.py",
        "--config", cfg,
        "--checkpoint", ckpt,
        "--split", "test",
        "--batch-size", "8",
    ])

for name in DATASET_NAMES + [COMBINED_NAME]:
    eval_model(name)



# Merge models (mean, closed-form, individual-fixed)

In [ ]:
import os, time
from pathlib import Path

os.makedirs("../models/merged", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("results", exist_ok=True)

CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"

CKPTS = {name: f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt" for name in DATASET_NAMES}
TEACHER_NAMES = list(DATASET_NAMES)
TEACHER_CHECKPOINTS = [CKPTS[name] for name in TEACHER_NAMES]
TEACHER_CONFIGS = [f"../models/trained/{name}_{MODEL_SIZE}/{name}_config.json" for name in TEACHER_NAMES]
TEACHER_DATASETS_SHARED = [str(COMBINED_DB)] * len(TEACHER_NAMES)
TEACHER_DATASETS_SEPARATE = [str(DATASET_DBS[name]) for name in TEACHER_NAMES]

MERGE_TIMES = {}

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_mean.py",
    "--config", CONFIG,
    "--output", "../models/merged/merged_mean.ckpt", "--keep-metadata",
]
for ckpt in TEACHER_CHECKPOINTS:
    cmd += ["--teacher", ckpt]
run(cmd)
MERGE_TIMES["merged_mean"] = time.time() - start

# Advanced baselines (after mean merge, before closed-form)
MERGE_SAMPLES = 1000
SOLVE_WORKERS = max(1, min(8, os.cpu_count() or 1))

MODEL_SIZE_TO_BASE = {
    "tiny": "orb-d3-xs-v2",
    "small": "orb-d3-sm-v2",
    "full": "orb-v2",
}
BASE_MODEL = MODEL_SIZE_TO_BASE[MODEL_SIZE]

PRETRAINED_CKPT = f"../models/pretrained/{BASE_MODEL}_pretrained.ckpt"
os.makedirs("../models/pretrained", exist_ok=True)
if not Path(PRETRAINED_CKPT).exists():
    run([
        "python", "-u", "../scripts/save_pretrained_orb.py",
        "--model", BASE_MODEL,
        "--output", PRETRAINED_CKPT,
    ])

TIES_OUTPUT = "../models/merged/merged_ties.ckpt"
TIES_GRID_RESULTS = "../results/ties_grid_search.json"

start = time.time()
ties_cmd = [
    "python", "-u", "../scripts/ties_merging_orb.py",
    "--pretrained-checkpoint", PRETRAINED_CKPT,
    "--output", TIES_OUTPUT,
    "--base-model", BASE_MODEL,
    "--grid-search",
    "--val-config", CONFIG,
    "--val-dataset", str(COMBINED_DB),
    "--density-values", "0.05", "0.1", "0.2", "0.3", "0.4",
    "--lambda-values", "0.3", "0.5", "0.8", "1.0", "1.2",
    "--metric", "val_Force_MAE",
    "--batch-size", "8",
    "--force-only",
    "--save-grid-results", TIES_GRID_RESULTS,
]
for ckpt in TEACHER_CHECKPOINTS:
    ties_cmd += ["--checkpoint", ckpt]
run(ties_cmd)
MERGE_TIMES["merged_ties"] = time.time() - start

FISHER_OUTPUT = "../models/merged/merged_fisher.ckpt"
FISHER_LOG = "../logs/fisher_merge.jsonl"

start = time.time()
fisher_cmd = [
    "python", "-u", "../scripts/fisher_merge_orb.py",
    "--base-model", BASE_MODEL,
    "--db-path", str(COMBINED_DB),
    "--dataset-name", COMBINED_NAME,
    "--config", CONFIG,
    "--output-ckpt", FISHER_OUTPUT,
    "--num-samples", str(MERGE_SAMPLES),
    "--batch-size", "8",
    "--force-only",
    "--normalize-fishers",
    "--data-units", DATA_UNITS,
    "--log-path", FISHER_LOG,
]
for ckpt in TEACHER_CHECKPOINTS:
    fisher_cmd += ["--checkpoint", ckpt]
run(fisher_cmd)
MERGE_TIMES["merged_fisher"] = time.time() - start

EMR_OUTPUT_DIR = "../models/merged/emr"
EMR_LOG = "../logs/emr_merge.jsonl"
os.makedirs(EMR_OUTPUT_DIR, exist_ok=True)

start = time.time()
emr_cmd = [
    "python", "-u", "../scripts/emr_merge_orb.py",
    "--pretrained-checkpoint", PRETRAINED_CKPT,
    "--output-dir", EMR_OUTPUT_DIR,
    "--base-model", BASE_MODEL,
    "--batch-size", "8",
    "--force-only",
    "--save-unified",
    "--evaluate",
    "--log-path", EMR_LOG,
]
for ckpt, cfg in zip(TEACHER_CHECKPOINTS, TEACHER_CONFIGS):
    emr_cmd += ["--checkpoint", ckpt]
    emr_cmd += ["--config", cfg]
run(emr_cmd)
MERGE_TIMES["emr_unified"] = time.time() - start
EMR_OUTPUT = f"{EMR_OUTPUT_DIR}/emr_unified.ckpt"

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_closed_form.py",
    "--config", CONFIG,
    "--dataset", str(COMBINED_DB),
    "--output", "../models/merged/merged_closed_form.ckpt",
    "--regularization", "1e-6", "--batch-size", "8",
]
for ckpt in CKPTS.values():
    cmd += ["--teacher", ckpt]
run(cmd)
MERGE_TIMES["merged_closed_form"] = time.time() - start

start = time.time()
cmd = [
    "python", "-u", "../scripts/merge_closed_form_individual_fixed.py",
    "--output", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--regularization", "1e-6", "--batch-size", "64", "--limit", "500", "--solve-workers", str(SOLVE_WORKERS),
]
for name in DATASET_NAMES:
    cfg = f"../models/trained/{name}_{MODEL_SIZE}/{name}_config.json"
    ckpt = CKPTS[name]
    db = DATASET_DBS[name]
    cmd += ["--teacher", f"{name}|{cfg}|{ckpt}|{db}"]
merge_output = run_capture(cmd)
logged = parse_logged_time(
    merge_output,
    r"Closed-form individual merge compute time:\s*([0-9.]+)s",
)
MERGE_TIMES["merged_closed_form_individual"] = (
    logged if logged is not None else (time.time() - start)
)

# Evaluation of Merged Models

In [ ]:
from pathlib import Path

CONFIG = f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json"
DATA = str(COMBINED_DB)

EVAL_CHECKPOINTS = {
    "merged_mean": "../models/merged/merged_mean.ckpt",
    "merged_ties": TIES_OUTPUT,
    "merged_fisher": FISHER_OUTPUT,
    "merged_closed_form": "../models/merged/merged_closed_form.ckpt",
    "merged_closed_form_individual_new": "../models/merged/merged_closed_form_individual_new.ckpt",
}

if "EMR_OUTPUT" in globals() and Path(EMR_OUTPUT).exists():
    EVAL_CHECKPOINTS["emr_unified"] = EMR_OUTPUT


def eval_ckpt(name, ckpt):
    run([
        "python", "-u", "../scripts/evaluate_model.py",
        "--config", CONFIG,
        "--checkpoint", ckpt,
        "--dataset", DATA,
        "--split", "test",
        "--batch-size", "8",
        "--save", f"../results/{name}_eval.txt",
    ])

for name, ckpt in EVAL_CHECKPOINTS.items():
    eval_ckpt(name, ckpt)


# Switch-embedding Evaluation of Individual-Closed-Form-Merge

In [ ]:
cmd = [
    "python", "-u", "../scripts/evaluate_switch_embeddings.py",
    "--config", f"../models/trained/{COMBINED_NAME}_{MODEL_SIZE}/{COMBINED_NAME}_config.json",
    "--checkpoint", "../models/merged/merged_closed_form_individual_new.ckpt",
    "--dataset", str(COMBINED_DB),
]
for name in DATASET_NAMES:
    ckpt = f"../models/trained/{name}_{MODEL_SIZE}/{name}_best.ckpt"
    cmd += ["--source-checkpoint", f"{name}={ckpt}"]
cmd += ["--split", "test", "--batch-size", "1", "--save", "../results/merged_closed_form_individual_switch_eval.txt"]
run(cmd)



In [ ]:
import csv, re
from pathlib import Path


def _parse_key(text: str, key: str):
    match = re.search(rf"^\s*{re.escape(key)}:\s*([0-9.eE+-]+)\s*$", text, re.M)
    if match:
        return float(match.group(1))
    return None


def parse_force_metrics(path: Path):
    if not path.exists():
        return None, None
    text = path.read_text()
    match = re.search(r"Forces -> MAE: ([0-9.eE+-]+), RMSE: ([0-9.eE+-]+)", text)
    if match:
        return float(match.group(1)), float(match.group(2))
    mae = _parse_key(text, "raw_forces_mae") or _parse_key(text, "forces_mae") or _parse_key(text, "force_mae")
    rmse = _parse_key(text, "raw_forces_rmse") or _parse_key(text, "forces_rmse") or _parse_key(text, "force_rmse")
    return mae, rmse


def format_time(value):
    if value is None:
        return ""
    try:
        return f"{float(value):.2f}"
    except (TypeError, ValueError):
        return str(value)


def write_summary_row(writer, seed, name: str, eval_path: Path, time_sec):
    mae, rmse = parse_force_metrics(eval_path)
    writer.writerow([
        "" if seed is None else seed,
        name,
        "" if mae is None else f"{mae:.6f}",
        "" if rmse is None else f"{rmse:.6f}",
        format_time(time_sec),
    ])

seed_value = globals().get("TRAIN_SEED", globals().get("SEED"))

dataset_names = globals().get("DATASET_NAMES")
if dataset_names:
    dataset_rows = list(dataset_names) + [COMBINED_NAME]
else:
    dataset_rows = [DATASET_A_NAME, DATASET_B_NAME, COMBINED_NAME]

train_times = globals().get("TRAIN_TIMES", {})
merge_times = globals().get("MERGE_TIMES", {})

rows = []
output_path = Path(f"baselines_summary{CSV_SUFFIX}.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

with output_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.writer(handle)
    writer.writerow(["seed", "name", "force_mae", "force_rmse", "time_sec"])

    for name in dataset_rows:
        eval_path = Path("results") / f"{name}_best_eval.txt"
        write_summary_row(writer, seed_value, name, eval_path, train_times.get(name))

    write_summary_row(
        writer,
        seed_value,
        "merged_mean",
        Path("results") / "merged_mean_eval.txt",
        merge_times.get("merged_mean"),
    )
    write_summary_row(
        writer,
        seed_value,
        "merged_ties",
        Path("results") / "merged_ties_eval.txt",
        merge_times.get("merged_ties"),
    )
    write_summary_row(
        writer,
        seed_value,
        "merged_fisher",
        Path("results") / "merged_fisher_eval.txt",
        merge_times.get("merged_fisher"),
    )
    write_summary_row(
        writer,
        seed_value,
        "emr_unified",
        Path("results") / "emr_unified_eval.txt",
        merge_times.get("emr_unified"),
    )
    write_summary_row(
        writer,
        seed_value,
        "merged_closed_form",
        Path("results") / "merged_closed_form_eval.txt",
        merge_times.get("merged_closed_form"),
    )
    write_summary_row(
        writer,
        seed_value,
        "merged_closed_form_individual",
        Path("results") / "merged_closed_form_individual_new_eval.txt",
        merge_times.get("merged_closed_form_individual"),
    )
    write_summary_row(
        writer,
        seed_value,
        "merged_closed_form_individual_switch",
        Path("results") / "merged_closed_form_individual_switch_eval.txt",
        None,
    )

print(f"Wrote {output_path}")